# Proyecto M2 ABIGAIL ÁLVAREZ

## Avance 3 — Data Warehouse y ETL en Snowflake

**Carrera:** Data Science — Henry

**Proyecto:** FleetLogix

### Objetivo
Construir un Data Warehouse con modelo estrella en Snowflake, integrando datos de PostgreSQL mediante un pipeline ETL desarrollado en Python.


In [1]:
import psycopg2
print("Librería PostgreSQL disponible")

Librería PostgreSQL disponible


In [2]:
import psycopg2
from getpass import getpass

conexion = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="fleetlogix",
    user="fleetlogix_user",
    password=getpass("Contraseña de PostgreSQL: ")
)

print("✅ Conexión exitosa con FleetLogix")
conexion.close()

Contraseña de PostgreSQL:  ········


✅ Conexión exitosa con FleetLogix


In [3]:
import pandas as pd

conexion = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="fleetlogix",
    user="fleetlogix_user",
    password=getpass("Contraseña de PostgreSQL: ")
)

tablas = ["vehicles", "drivers", "routes", "trips", "deliveries", "maintenance"]

for tabla in tablas:
    df = pd.read_sql_query(f"SELECT COUNT(*) AS total FROM {tabla}", conexion)
    print(f"{tabla}: {df['total'][0]:,} registros")

conexion.close()

Contraseña de PostgreSQL:  ········


/tmp/ipykernel_14836/2925426056.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT COUNT(*) AS total FROM {tabla}", conexion)


vehicles: 200 registros
drivers: 400 registros
routes: 50 registros
trips: 100,000 registros
deliveries: 400,000 registros
maintenance: 5,000 registros


## Arquitectura del Data Warehouse

**Flujo de datos:** PostgreSQL → Python ETL → Snowflake STAGING → MERGE incremental → FACT_DELIVERIES → Indicadores KPI.

**Modelo dimensional:** una tabla de hechos (FACT_DELIVERIES) y seis dimensiones: cliente, fecha, conductor, ruta, hora y vehículo.

**Granularidad:** una fila por entrega.

El modelo y sus relaciones están documentados en `docs/MODELO_ESTRELLA.md`.

## Pipeline ETL en Python

El proceso de integración está dividido en cuatro etapas:

1. **Extracción:** `etl_extract_deliveries.py` obtiene los datos desde PostgreSQL.
2. **Staging:** `etl_refresh_staging.py` prepara los registros en Snowflake.
3. **Carga incremental:** `etl_merge_deliveries.py` actualiza la tabla de hechos mediante MERGE.
4. **Calidad de datos:** `etl_quality_checks.py` ejecuta las validaciones automáticas.

**Orquestador principal:** `etl_pipeline.py`.

Los scripts se encuentran en la raíz del repositorio y permiten separar las responsabilidades de cada etapa.

## Indicadores KPI — FleetLogix

Resultados obtenidos en la validación de Snowflake:

| Indicador | Resultado |
|---|---:|
| Total de entregas | 400.000 |
| Entregas completadas | 339.820 |
| Entregas pendientes | 60.180 |
| Entregas puntuales | 48.299 |
| Entregas atrasadas | 291.521 |
| Porcentaje de puntualidad | 14,21 % |
| Demora promedio | 180,37 minutos |

**Resultado de la validación:** OK.

Estos valores corresponden a una ejecución anterior; esta celda no consulta Snowflake en tiempo real.

## Validación y calidad de datos

Resultados de la auditoría realizada en Snowflake:

| Control | Resultado |
|---|---:|
| Total de entregas | 400.000 |
| Entregas puntuales | 48.299 |
| Entregas atrasadas | 291.521 |
| Entregas pendientes | 60.180 |
| Pendientes inconsistentes | 0 |
| Completadas sin hora | 0 |

**Resultado:** validación de indicadores OK.

Se verificó que las categorías de entregas suman 400.000 registros y que no existen inconsistencias en los dos controles de estado mostrados.

Estos resultados corresponden a una ejecución anterior, no a una consulta en tiempo real.